# Member 3 - Autoencoder / Unknown-Attack Detection

## Work completed through Day 4
This notebook is intentionally limited to the work that should be complete by the end of Day 4.

- **Day 1:** define the unknown-attack detection approach and held-out attacks.
- **Day 2:** prepare the notebook/project structure and reproducibility settings.
- **Day 3:** define the BENIGN-only training requirement and data contract with Member 1.
- **Day 4:** load Member 1's final dataset, verify it, extract BENIGN traffic and the held-out UNKNOWN attacks, and prepare the data subsets needed for later Autoencoder work.

**No Autoencoder training is performed yet.** Training begins after the final dataset is received and Day-4 checks pass.


In [ ]:
# Day 2 - Imports and reproducibility
from pathlib import Path
import random
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

pd.set_option("display.max_columns", 100)
print("Member 3 setup ready.")


In [ ]:
# Day 2 - Project paths
# This notebook is expected at: notebooks/autoencoder.ipynb
PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "outputs" / "results"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
MODELS_DIR = PROJECT_ROOT / "saved-models"

for folder in (DATA_DIR, RESULTS_DIR, FIGURES_DIR, MODELS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

# Change only this filename if Member 1 uses a different name.
DATA_PATH = DATA_DIR / "unknown_test_Bot.csv"

print("Project root :", PROJECT_ROOT.resolve())
print("Expected data:", DATA_PATH.resolve())


## Day 1 - Experiment definition
The Autoencoder will later be trained **only on BENIGN traffic**. The selected attacks below are held out from Autoencoder training and treated collectively as **UNKNOWN** during the final anomaly-detection experiment. Their original labels are retained only for research/evaluation.


In [ ]:
# Day 1 - Define held-out UNKNOWN attacks
NORMAL_LABEL = "BENIGN"

UNKNOWN_ATTACKS = [
    "Bot",
    "Infiltration",
    "Web Attack - SQL Injection",
    "Heartbleed",
]

print("Normal label:", NORMAL_LABEL)
print("Held-out UNKNOWN attacks:")
for attack in UNKNOWN_ATTACKS:
    print(" -", attack)


## Day 3 - Data contract with Member 1
Member 1 is responsible for the common final dataset and its cleaning/feature-selection policy. Member 3 does not independently redo that cleaning.

The final dataset must contain:
- a `Label` column;
- `BENIGN` records;
- the selected held-out attack records;
- numeric model features after Member 1's preparation;
- no unresolved NaN or infinite values in model features.

`source_file` may remain as metadata, but it will not be used as a model feature.


In [ ]:
# Day 4 - Load Member 1's final dataset
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Final dataset not found at: {DATA_PATH}\n"
        "When Member 1 provides the file, place it in data/processed/ "
        "or update DATA_PATH above."
    )

suffix = DATA_PATH.suffix.lower()
if suffix == ".parquet":
    data = pd.read_parquet(DATA_PATH)
elif suffix == ".csv":
    data = pd.read_csv(DATA_PATH, low_memory=False)
else:
    raise ValueError("Final dataset must be .parquet or .csv")

print(f"Loaded: {DATA_PATH.name}")
print(f"Rows   : {len(data):,}")
print(f"Columns: {data.shape[1]}")


In [ ]:
# Day 4 - Basic schema and label checks
if "Label" not in data.columns:
    raise KeyError("Required column 'Label' is missing from Member 1's dataset.")

print("Column names:")
print(data.columns.tolist())

print("\nClass distribution:")
label_counts = data["Label"].value_counts(dropna=False)
display(label_counts.to_frame("count"))

if NORMAL_LABEL not in set(data["Label"].dropna()):
    raise ValueError(f"Normal label '{NORMAL_LABEL}' was not found.")

missing_unknown = [a for a in UNKNOWN_ATTACKS if a not in set(data["Label"].dropna())]
if missing_unknown:
    print("WARNING - these configured UNKNOWN labels are missing:", missing_unknown)
else:
    print("All configured UNKNOWN attack labels are present.")


In [ ]:
# Day 4 - Extract only the data needed by Member 3
normal_data = data.loc[data["Label"] == NORMAL_LABEL].copy()
unknown_data = data.loc[data["Label"].isin(UNKNOWN_ATTACKS)].copy()

if normal_data.empty:
    raise ValueError("No BENIGN records were extracted.")
if unknown_data.empty:
    raise ValueError("No held-out UNKNOWN attack records were extracted.")

print(f"BENIGN records  : {len(normal_data):,}")
print(f"UNKNOWN records : {len(unknown_data):,}")
print("\nUNKNOWN composition (evaluation metadata only):")
display(unknown_data["Label"].value_counts().to_frame("count"))


In [ ]:
# Day 4 - Define model feature columns and verify Member 1's final data
NON_FEATURE_COLUMNS = [c for c in ["Label", "source_file"] if c in data.columns]
feature_columns = [c for c in data.columns if c not in NON_FEATURE_COLUMNS]

non_numeric = [c for c in feature_columns if not pd.api.types.is_numeric_dtype(data[c])]
if non_numeric:
    raise TypeError(
        "Non-numeric model features remain in the final dataset: " + str(non_numeric)
    )

model_values = data[feature_columns].to_numpy(dtype=np.float64)
nan_count = int(np.isnan(model_values).sum())
inf_count = int(np.isinf(model_values).sum())

print(f"Model features : {len(feature_columns)}")
print(f"NaN values     : {nan_count:,}")
print(f"Infinite values: {inf_count:,}")

if nan_count or inf_count:
    raise ValueError(
        "Member 1's final model features still contain NaN/Inf values. "
        "Resolve this in the common preprocessing pipeline before Autoencoder training."
    )

print("Day-4 data-quality checks passed.")


In [ ]:
# Day 4 - Prepare Member 3 subsets for the next stage
# IMPORTANT: This cell does NOT train the model.
# It only prepares the BENIGN and UNKNOWN feature matrices.

X_normal = normal_data[feature_columns].copy()
X_unknown = unknown_data[feature_columns].copy()

# Keep original attack names separately for research-only evaluation.
unknown_evaluation_labels = unknown_data["Label"].copy()

print("X_normal shape :", X_normal.shape)
print("X_unknown shape:", X_unknown.shape)
print("Evaluation labels retained separately:", len(unknown_evaluation_labels))


In [ ]:
# Day 4 - Save a small preparation summary (not the full dataset)
summary = pd.DataFrame({
    "item": [
        "total_final_dataset_rows",
        "total_features_for_member3",
        "benign_rows",
        "held_out_unknown_rows",
    ],
    "value": [
        len(data),
        len(feature_columns),
        len(normal_data),
        len(unknown_data),
    ],
})

summary_path = RESULTS_DIR / "member3_day4_data_summary.csv"
summary.to_csv(summary_path, index=False)
display(summary)
print("Saved:", summary_path)


## End of Day 4
At this point Member 3 has:
1. fixed the unknown-attack experiment definition;
2. prepared the project paths and notebook structure;
3. defined the required data contract;
4. loaded and verified Member 1's final dataset;
5. extracted BENIGN traffic for future Autoencoder development;
6. extracted the selected attacks as held-out UNKNOWN data; and
7. verified that the model features are numeric and contain no NaN/Inf values.

**Next (Day 5 onward):** create the Autoencoder development split, fit preprocessing only where required, build/train the Autoencoder, select the reconstruction-error threshold, and evaluate on NORMAL vs held-out UNKNOWN traffic.
